In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

import numpy  as np
import pandas as pd
import h5py
from pathlib import Path

from astropy    import units as u
import matplotlib
from matplotlib import pyplot as plt
import matplotlib.patches as mpatches
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'

font = {'family' : 'sans-serif',
        'size'   : 8}

matplotlib.rc('font', **font)

from scipy.interpolate import RegularGridInterpolator
from scipy.signal      import argrelextrema

EHT = os.path.abspath("../2017_sgra_paper5")
RAPTOR = os.path.abspath("../RAPTOR")
KNS = os.path.abspath("../mod")

sys.path.append(EHT)
    
from common import hallmark as hm
from common import mockservation as mk
from common import io_raptor as io
from common import dalt
from common import viz

sys.path.remove(EHT)

sys.path.append(RAPTOR)
from python.plotting import rapplot
sys.path.remove(RAPTOR)

sys.path.append(KNS)
import module
import plotscripts 
from plotscripts import subtract, image2d, loadfigure, getimagedata


In [2]:
def import_ab(a, i, cond):
    file = f'../cache/shadows/shadow_a{a:.2f}_i{i:g}.h5' # change this to your local folder
    with h5py.File(file) as h:
      alpha = h['a'][:]
      beta = h['b'][:]

    if cond:
      index_apos = np.where(alpha > 0)[0]
      aright = index_apos[beta[index_apos].argmin()]

      for bb in np.linspace(-10e-3, 10e-3, 10):
        alpha = np.append(alpha, alpha[aright])
        beta = np.append(beta, bb)

    return (alpha, beta)

## Load paths to images

In [3]:
# Load all images 
pf = hm.ParaFrame('../cache/{mag}a{aspin:g}_w{window:d}/Rh{Rhigh:d}_i{inc:d}/n{n:n}/img_data_{snapshot:d}.h5')
sel = pf(snapshot=[999, 1000, 101000])
display(sel)

/Users/suzukihina/Documents/UR/NkS/KNS_PR/2017_sgra_paper5/common/hallmark.py:26: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  mask |= self[k].isin(v)


,path,mag,aspin,window,Rhigh,inc,n,snapshot
0,../cache/Ma101_w1/Rh1_i15/n0/img_data_999.h5,M,101.0,1,1,15,0,999
1,../cache/Ma101_w1/Rh1_i15/n1/img_data_1000.h5,M,101.0,1,1,15,1,1000
2,../cache/Ma101_w1/Rh1_i15/n1/img_data_101000.h5,M,101.0,1,1,15,1,101000
3,../cache/Ma101_w1/Rh1_i15/n1/img_data_999.h5,M,101.0,1,1,15,1,999
4,../cache/Ma101_w1/Rh1_i15/n100/img_data_1000.h5,M,101.0,1,1,15,100,1000
5,../cache/Ma101_w1/Rh1_i15/n100/img_data_999.h5,M,101.0,1,1,15,100,999
6,../cache/Ma101_w1/Rh1_i15/n2/img_data_1000.h5,M,101.0,1,1,15,2,1000
7,../cache/Ma101_w1/Rh1_i15/n2/img_data_101000.h5,M,101.0,1,1,15,2,101000
8,../cache/Ma101_w1/Rh1_i15/n2/img_data_999.h5,M,101.0,1,1,15,2,999
9,../cache/Ma101_w1/Rh1_i30/n0/img_data_999.h5,M,101.0,1,1,30,0,999


In [4]:
ns = [100, 0, 1,2]
incs = np.sort(np.array(sel.inc.unique())) 

## Create a mask 

In [5]:
#get critical curve 
alpha1, beta1 = import_ab(1.01, 15, True) 
# alpha1

In [6]:
N = 1000
xmin, xmax = -20, 20

x = np.linspace(xmin, xmax, N)
y = np.linspace(xmin, xmax, N)

X, Y = np.meshgrid(x, y)

In [7]:
valid = np.isfinite(alpha1) & np.isfinite(beta1)
alpha = alpha1[valid]
beta  = beta1[valid]

In [8]:
from matplotlib.path import Path
curve = Path(np.column_stack((alpha, beta)))
points = np.column_stack((X.ravel(), Y.ravel()))

# TODO: optimize 
mask = curve.contains_points(points).reshape(N, N)

In [18]:
plt.imshow(mask, extent=[-20, 20, -20, 20], origin='lower')
plt.plot(alpha1, beta1, 'r-', lw=1)
plt.xlabel(r'$\alpha$')
plt.ylabel(r'$\beta$')
# plt.show()

/var/folders/bc/b_svn48x29l_b6t84kwwg60w0000gn/T/ipykernel_3648/541607343.py:1: UserWarning: Attempt to set non-positive ylim on a log-scaled axis will be ignored.
  plt.imshow(mask, extent=[-20, 20, -20, 20], origin='lower')


Text(42.0, 0.5, '$\\beta$')

## Apply mask to images 

In [10]:
inc = 15 
n = 1


from pathlib import Path
s = sel[(sel['inc'] == inc) & (sel['n'] == n) & (sel['snapshot'] == 101000)]
path_object = Path(s.path.item())
img_path = str(path_object.parent)
img_path

'../cache/Ma101_w1/Rh1_i15/n1'

In [11]:
fig, ax = plt.subplots(1, 1)
data_id_all, min, max, image_array = getimagedata(img_path, 1000)
figure = loadfigure(image_array, np.min(image_array), np.max(image_array), fig, ax, 20, 1, label="Stokes", cmap="afmhot")


Reading keys from:  ../cache/Ma101_w1/Rh1_i15/n1/img_data_1000.h5
['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']
Reading in:  ../cache/Ma101_w1/Rh1_i15/n1/img_data_1000.h5
<KeysViewHDF5 ['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']>
8


In [12]:
print("masking inner region to be 0")
oimg = image_array.copy()
oimg[mask] = 0

masking inner region to be 0


In [13]:
print("masking outer region to be 0")
iimg = image_array.copy()
iimg[~mask] = 0

masking outer region to be 0


In [14]:

fig, axes = plt.subplots(1, 3)

im1 = loadfigure(image_array, np.min(image_array), np.max(image_array), fig, axes[0], 20, 1, label="Full image", cmap="afmhot")
im2 = loadfigure(oimg, np.min(oimg), np.max(oimg), fig, axes[1], 20, 1, label="KNS_outside", cmap="afmhot")
im3 = loadfigure(iimg, np.min(iimg), np.max(iimg), fig, axes[2], 20, 1, label="KNS_inside", cmap="afmhot")

axes[0].set_title("Full image")
axes[1].set_title("KNS_outside")
axes[2].set_title("KNS_inside")

fig.savefig(f"../plot/fig_inout_i{inc}.pdf")

In [15]:
from pathlib import Path

# mask out inner region. BH 
mov_omask = io.load_mov(s.path, [1000], 20, mask= mask, resize = False)

# mask out outer region. NKS - BH
mov_imask = io.load_mov(s.path,[1000], 20, mask= ~mask, resize = False)

# original image
mov = io.load_mov(s.path, [1000], 20, mask=None, resize = False)

In [16]:
fig, axes = plt.subplots(1, 1)

h1 = module.plot_va(mov_omask, axes, n, i=45)[0]
h2 = module.plot_va(mov_imask, axes, n, i=45)[0]
h3 = module.plot_va(mov, axes, n, i=45)[0]

h1.set_label("KNS_outside")
h2.set_label("KNS_inside")
h3.set_label("KNS_full")

# axes.set_xlim(0,100) 
fig.legend(loc='upper right')

fig.savefig(f"../plot/va_inout_i{inc}.pdf")


In [17]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6), sharey=True, sharex=True)

inc = 15
i = 0

legend_handles = []  

for col, n in enumerate(ns): 
    if n in [1, 2]: 
        s = sel[(sel['inc'] == inc) & (sel['n'] == n) & (sel['snapshot'] == 1000)]
        path_object = Path(s.path.item())
        mov_omask = io.load_mov(s.path, [1000], 20, mask= mask, resize = True)
 
    else: 
        s = sel[(sel['inc'] == inc) & (sel['n'] == n) & (sel['snapshot'] == 999)]
        path_object = Path(s.path.item())
        mov_omask = io.load_mov(s.path, [900], 20, mask= mask, resize = True)
    
    # Plot VA
    handles = module.plot_va(mov_omask, ax, n, i)
    legend_handles.extend(handles) 
    
    # add lines for EHT, BHEX 
    ax.axvline(x=8.6, color='black', linestyle='--', lw=0.8)
    ax.axvline(x=35, color='black', linestyle='--', lw=0.8)
    
    ax.set_xlim(0,200)
    

fig.legend(
    handles=legend_handles[0:4],
    fontsize=10,
    ncol=4,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.9),
    frameon=True
)

fig.savefig(f"../plot/va_all_i{inc}.pdf")
